# TRỰC QUAN HÓA DỮ LIỆU & KHÁM PHÁ (EXPLORATORY DATA ANALYSIS - EDA)

Notebook này tập trung vào việc tạo các biểu đồ trực quan hóa dữ liệu chuẩn xuất bản báo cáo (300 DPI, tối giản viền theo chuẩn Springer) nhằm giải quyết các câu hỏi nghiên cứu (**RQ1 & RQ2**).

## Cell 1: Khai báo thư viện & Cấu hình giao diện chuẩn (Style Config)

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
sns.set_style('ticks')

os.makedirs('report', exist_ok=True)

def clean_spines(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

print("✓ Khai báo thư viện và cấu hình đồ thị thành công.")

## Cell 2: Tải dữ liệu đã làm sạch từ DuckDB

In [ ]:
con = duckdb.connect('data/processed/ady.duckdb')
df = con.execute("SELECT * FROM feat").df()

if 'y_encoded' not in df.columns and 'y' in df.columns:
    df['y_encoded'] = df['y'].astype('category').cat.codes

print(f"✓ Tải thành công {len(df):,} bản ghi từ DuckDB.")

## Cell 3: Hình 1 - Phân phối biến mục tiêu (Histogram + Boxplot)
> **Mục tiêu:** Kiểm tra độ lệch (skewness) và phát hiện các điểm ngoại lệ (outliers) của biến mục tiêu.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))

ax[0].hist(df['y_encoded'], bins=15, color='#1B6B6D', edgecolor='white')
ax[0].set_xlabel('Target Category (Encoded)', fontsize=10)
ax[0].set_ylabel('Incident Count', fontsize=10)
ax[0].set_title('Frequency Distribution', fontsize=11, fontweight='bold')
clean_spines(ax[0])

sns.boxplot(x=df['y_encoded'], color='#9FBFBF', ax=ax[1], showfliers=True)
ax[1].set_xlabel('Target Category (Encoded)', fontsize=10)
ax[1].set_title('Target Dispersion & Outliers', fontsize=11, fontweight='bold')
clean_spines(ax[1])

fig.tight_layout()
fig.savefig('report/fig_rq1_distribution.png', bbox_inches='tight')
plt.show()

print("✓ Saved: report/fig_rq1_distribution.png")

## Cell 4: Hình 2 - Biểu đồ cột so sánh theo Năm kèm số liệu in đậm (Grouped Bar Chart)
> **Mục tiêu:** So sánh xu hướng sự cố giữa các năm và hiển thị số liệu cụ thể trên đầu mỗi cột.

In [ ]:
yearly = df.groupby('year')['y_encoded'].count().reset_index()

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(yearly['year'].astype(str), yearly['y_encoded'], color='#1B6B6D')

ax.bar_label(bars, fmt='%d', fontweight='bold', fontsize=8, padding=3)

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Total Reported Incidents', fontsize=10)
ax.set_title('Figure 1: Annual UAS Incident Trend', fontsize=11, fontweight='bold')
clean_spines(ax)

fig.tight_layout()
fig.savefig('report/fig_rq1_period_bar.png', bbox_inches='tight')
plt.show()

print("✓ Saved: report/fig_rq1_period_bar.png")

## Cell 5: Hình 3 - Ma trận nhiệt thời gian x Loại vùng trời (Heatmap Period x Airspace)
> **Mục tiêu:** Trực quan hóa điểm nóng sự cố theo thời gian và phân loại vùng trời.

In [ ]:
top_airspaces = df['airspace'].value_counts().head(8).index
df_hm = df[df['airspace'].isin(top_airspaces)]
pivot_hm = df_hm.pivot_table(index='airspace', columns='year', values='y_encoded', aggfunc='count').fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot_hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Incident Count'})

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Airspace Class', fontsize=10)
ax.set_title('Figure 2: Heatmap of Incident Concentration by Airspace & Year', fontsize=11, fontweight='bold')

fig.tight_layout()
fig.savefig('report/fig_rq1_heatmap.png', bbox_inches='tight')
plt.show()

print("✓ Saved: report/fig_rq1_heatmap.png")

## Cell 6: Hình 4 - Biểu đồ tương quan giữa các đặc trưng (Correlation Heatmap)
> **Mục tiêu:** Phát hiện đa cộng tuyến giữa các biến ngoại sinh và độ dài đoạn văn bản mô tả.

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns
corr_matrix = df[num_cols].corr(method='spearman').round(2)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax, cbar=True)

ax.set_title('Figure 3: Feature Correlation Matrix (Spearman)', fontsize=11, fontweight='bold')
fig.tight_layout()
fig.savefig('report/fig_rq1_corr.png', bbox_inches='tight')
plt.show()

print("✓ Saved: report/fig_rq1_corr.png")

## Cell 7: Hình 5 - Phân bố sự cố theo Giai đoạn bay (Flight Phase Distribution)
> **Mục tiêu:** Xác định giai đoạn bay có nguy cơ xảy ra sự cố cao nhất (Cruise vs Approach/Landing).

In [ ]:
phase_counts = df['flight_phase'].value_counts().head(7).reset_index()
phase_counts.columns = ['flight_phase', 'count']

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(phase_counts['flight_phase'], phase_counts['count'], color='#2E86AB')

ax.bar_label(bars, fmt='%d', fontweight='bold', fontsize=8, padding=3)

ax.set_xlabel('Incident Count', fontsize=10)
ax.set_ylabel('Flight Phase', fontsize=10)
ax.set_title('Figure 4: Top High-Risk Flight Phases', fontsize=11, fontweight='bold')
ax.invert_yaxis()  
clean_spines(ax)

fig.tight_layout()
fig.savefig('report/fig_rq1_flight_phase.png', bbox_inches='tight')
plt.show()

print("✓ Saved: report/fig_rq1_flight_phase.png")

## Cell 8: Nhật ký AI Audit Log (Minh chứng Human Delta cho Bước 5)

### AI Audit Log - Step 5: Data Visualization
* **Prompt được sử dụng:** "Viết code matplotlib/seaborn vẽ 5 biểu đồ đáp ứng RQ1 cho báo cáo môn học."
* **Sự cố / Lỗi phát hiện được:** AI ban đầu tạo biểu đồ giữ nguyên viền trên/phải, thiếu nhãn giá trị trên đầu các cột (bar annotations) và truy cập sai chỉ số mảng subplot (`ax[6]` thay vì `ax[1]`).
* **Human Delta (Can thiệp của sinh viên):**
  1. Thêm hàm `clean_spines()` loại bỏ viền trên/phải giúp giao diện tối giản theo chuẩn Springer.
  2. Bổ sung `ax.bar_label()` để hiển thị chính xác con số dữ liệu trên biểu đồ cột.
  3. Sửa lỗi truy vết subplot chỉ số `ax[1]` và cấu hình độ phân giải cố định 300 DPI để không bị mờ khi chèn vào bài viết LaTeX.